<a href="https://colab.research.google.com/github/leoruggiero/Sprint-7-Python-Leonardo-Ruggiero/blob/main/Sprint7_Python_LeonardoRuggiero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nivel 1
## Ejercicio 1: Selección de Personal

Trabajas como **analista de datos** en el departamento de RRHH de una empresa. Se ha abierto una nueva oferta laboral interna y la dirección ha decidido que **sólo las primeras 10 personas** que se inscriban podrán participar en el proceso de selección.

Por ello, te han encargado desarrollar una herramienta sencilla en **Python** que actúe como formulario de inscripción y sistema de clasificación inicial.

---

### Requisitos del Formulario
El programa debe permitir introducir y validar los siguientes datos para cada candidato/a, asegurándose de que la información sea correcta y entre dentro de los valores permitidos:

1. **Nombre y Apellidos**
2. **Teléfono de contacto**
3. **Años de experiencia** dentro de la empresa
4. **Conocimiento de Catalán** (Escala 0-3):
   * `0` = Sin conocimiento
   * `1` = Hasta nivel B1
   * `2` = Nivel B2
   * `3` = C1 o superior
5. **Dominio de Excel** (Escala 0-3):
   * `0` = Nivel básico
   * `1` = Uso de funciones
   * `2` = Tablas dinámicas
   * `3` = Visual Basic y Macros
6. **Título académico** (Escala 0-3):
   * `0` = ESO
   * `1` = FP
   * `2` = Universidad
   * `3` = Máster o superior

---

### Lógica de Negocio y Clasificación

* **Cálculo de la Puntuación Total:**
  $$\text{Puntuación} = (\text{Años de experiencia} \times 1.5) + \text{Catalán} + \text{Excel} + \text{Titulación}$$

* **Umbrales de Clasificación:**
  *  **Prioridad Alta** $\rightarrow \text{Puntuación} \ge 12$
  *  **Candidato adecuado** $\rightarrow 8 \le \text{Puntuación} < 12$
  *  **No prioritario** $\rightarrow \text{Puntuación} < 8$

---

###  Funcionamiento Esperado
* **Flujo continuo:** Cada vez que alguien complete el formulario, el programa debe mostrar su puntaje individual inmediatamente.
* **Límite de registros:** Cuando se hayan inscrito **exactamente 10 personas**, la herramienta debe dejar de aceptar nuevos registros de forma automática.
* **Informe final:** Al finalizar, se debe generar un informe en pantalla ordenado con el siguiente criterio:
  1. Según la **prioridad** (de Alta a No prioritario).
  2. En caso de empate, por la **puntuación obtenida** (de mayor a menor).
* **Campos del informe:** El listado final debe incluir el *Nombre*, *Teléfono* y *Puntuación* de cada candidato para facilitar el contacto rápido por parte de RRHH.

### Funciones de Entrada y Validación de Datos

---

En la primeira etapa hago las funciones de solicitar texto y numero. Para asegurar que no falle durante su uso, he creado funciones auxiliares robustas para capturar las respuestas del usuario:
* En `solicitar_texto`, implementé un bucle `while True` que impide que el usuario deje campos obligatorios vacíos (como el nombre).
* En `solicitar_numero`, utilicé un bloque `try-except` para capturar el error `ValueError`. Si el usuario introduce letras en lugar de números, el programa no colapsa; simplemente devuelve un `-1`, lo cual me servirá como "bandera" para identificar datos inválidos en la siguiente fase.

In [19]:
LIMITE_CANDIDATOS = 10

def solicitar_texto(mensaje):
    while True:
        valor = input(mensaje).strip()
        if valor:
            return valor
        print("Error: El campo no puede estar vacío.")

def solicitar_numero(mensaje):
    valor = input(mensaje).strip()
    try:
        return int(valor)
    except ValueError:
        return -1

### Lógica de Negocio y Clasificación de Candidatos

---
En esta fase he programado la fórmula matemática exigida por dirección, pero añadiendo una capa extra de seguridad.
* En `calcular_puntuacion`, antes de sumar, verifico si algún valor está fuera de los rangos permitidos (ej. menor a 0 o mayor a 3). Si detecto una anomalía, fuerzo el resultado a `-1.0` para aislar ese registro.
* Posteriormente, la función `obtener_prioridad` traduce la puntuación numérica a las categorías ("Prioridad Alta", "Candidato adecuado", etc.). Además, le asigné un `peso` numérico (0 a 3) a cada categoría para facilitar el proceso de ordenamiento en el informe final.

In [20]:

def calcular_puntuacion(experiencia, catalan, excel, titulacion):
    if (catalan < 0 or catalan > 3 or
        excel < 0 or excel > 3 or
        titulacion < 0 or titulacion > 3 or
        experiencia < 0):
        return -1.0

    return (experiencia * 1.5) + catalan + excel + titulacion

def obtener_prioridad(puntuacion):
    if puntuacion == -1.0:
        return "ERROR: Datos Inválidos", 0  # Peso 0 para ordenar al final
    elif puntuacion >= 12:
        return "Prioridad Alta", 3
    elif 8 <= puntuacion < 12:
        return "Candidato adecuado", 2
    else:
        return "No prioritario", 1

### Función de Salida y Generación del Informe

---

Para generar el listado final ordenado por la doble condición que pide el enunciado, he optado por utilizar la función integrada `sorted()` combinada con una función `lambda`.
Esta es la solución me permite ordenar la lista priorizando primero el `'peso_prioridad'` y, en caso de empate, la `'puntuacion'` exacta, ambos de forma descendente (`reverse=True`).
Finalmente, itero sobre la lista ordenada para imprimirla con un formato tabular estructurado, sustituyendo cualquier puntuación errónea (`-1.0`) por un "N/A" para mantener la estética profesional del reporte.

In [21]:
def generar_informe(candidatos):
    print("\n\n" + "="*75)
    print("INFORME FINAL DE SELECCIÓN - DEPARTAMENTO DE RRHH")
    print("="*75)

    candidatos_ordenados = sorted(
        candidatos,
        key=lambda x: (x['peso_prioridad'], x['puntuacion']),
        reverse=True)

    print(f"{'Clasificación':<26} | {'Nombre y Apellidos':<20} | {'Teléfono':<10} | {'Puntos':<6}")
    print("-" * 75)

    for candidato in candidatos_ordenados:
        puntos_mostrar = "N/A" if candidato['puntuacion'] == -1.0 else candidato['puntuacion']

        print(f"{candidato['prioridad']:<26} | "
              f"{candidato['nombre'][:20]:<20} | "
              f"{candidato['telefono']:<10} | "
              f"{puntos_mostrar:<6}")
    print("="*75)

### Flujo Principal e Inicio del Formulario Interactivo

---

Para orquestar el funcionamiento continuo del programa, he encapsulado todo en la función `ejecutar_sistema_inscripcion_manual()`.
El "motor" de esta función es el bucle `while` cuya condición de salida es estrictamente la longitud de la lista de candidatos (`LIMITE_CANDIDATOS = 10`). Dentro del bucle, el programa recopila los datos, calcula e imprime la evaluación provisional en tiempo real, y empaqueta la información del empleado en un diccionario.
En cuanto la lista alcanza exactamente los 10 registros, el bucle finaliza automáticamente el proceso de inscripción y dispara la generación del informe.

In [22]:
def ejecutar_sistema_inscripcion_manual():
    lista_candidatos = []
    print("=== INSCRIPCTION MANUAL RRHH ===")

    while len(lista_candidatos) < LIMITE_CANDIDATOS:
        num_registro = len(lista_candidatos) + 1
        print(f"\n Registro de Candidato/a [{num_registro}/{LIMITE_CANDIDATOS}]:")

        nombre = solicitar_texto("Nombre y Apellidos: ")
        telefono = solicitar_texto("Teléfono de contacto: ")
        experiencia = solicitar_numero("Años de experiencia: ")
        catalan = solicitar_numero("Nivel Catalán (0-3): ")
        excel = solicitar_numero("Nivel Excel (0-3): ")
        titulacion = solicitar_numero("Título académico (0-3): ")

        puntos = calcular_puntuacion(experiencia, catalan, excel, titulacion)
        etiqueta, peso = obtener_prioridad(puntos)

        if puntos == -1.0:
            print(" Nota: El sistema detectó datos fuera de rango. El registro se archivará igual.")
        else:
            print(f" Resultado provisional: {puntos} puntos -> {etiqueta}")

        lista_candidatos.append({
            'nombre': nombre,
            'telefono': telefono,
            'puntuacion': puntos,
            'prioridad': etiqueta,
            'peso_prioridad': peso
        })

    print("\n Cupo máximo alcanzado (10/10). Formulario cerrado automáticamente.")
    generar_informe(lista_candidatos)

ejecutar_sistema_inscripcion_manual()

=== INSCRIPCTION MANUAL RRHH ===

 Registro de Candidato/a [1/10]:
Nombre y Apellidos: Laura Serra
Teléfono de contacto: 608112233
Años de experiencia: 6
Nivel Catalán (0-3): 3
Nivel Excel (0-3): 3
Título académico (0-3): 3
 Resultado provisional: 18.0 puntos -> Prioridad Alta

 Registro de Candidato/a [2/10]:
Nombre y Apellidos: Clara Gómez
Teléfono de contacto: 655221199
Años de experiencia: 3
Nivel Catalán (0-3): 3
Nivel Excel (0-3): 3
Título académico (0-3): 2
 Resultado provisional: 12.5 puntos -> Prioridad Alta

 Registro de Candidato/a [3/10]:
Nombre y Apellidos: Sergio López
Teléfono de contacto: 644112211
Años de experiencia: 3
Nivel Catalán (0-3): 2
Nivel Excel (0-3): 2
Título académico (0-3): 2
 Resultado provisional: 10.5 puntos -> Candidato adecuado

 Registro de Candidato/a [4/10]:
Nombre y Apellidos: Candidat Error
Teléfono de contacto: 611000000
Años de experiencia: 5
Nivel Catalán (0-3): 7
Nivel Excel (0-3): 3
Título académico (0-3): 2
 Nota: El sistema detectó datos f

## Ejercicio 2: Segmentación por riesgo de cesta de compra


Trabajas como analista de datos en el departamento de Marketing Digital. La empresa quiere mejorar la efectividad de las campañas de recuperación de cestas de compra abandonados y necesita identificar qué clientes tienen mayor riesgo de abandono, qué segmento de comportamiento presentan y qué mensaje publicitario es el más adecuado para cada uno.

Dispones de un conjunto de datos con información básica de 10 clientes:

* id_client
* nombre
* edad
* grupo_edad
* código_comportamiento

El departamento de CRM ha definido manualmente:

» Códigos de comportamiento
* '0' → Navega pero no añade productos
* '1' → Agregar productos pero no llega al checkout
* '2' → Llega al checkout pero no paga
* '3' → Cesta de compra abandonado hace menos de 24 h
* '4' → Cesta de compra abandonado hace más de 24 h

» Factores según código de comportamiento
* Código 0 → 0.10
* Código 1 → 0.25
* Código 2 → 0.50
* Código 3 → 0.75
* Código 4 → 0.90

» Factores según grupo de edad
* 18–25 → 1.20
* 26–40 → 1.00
* 41–60 → 0.85
* 60+ → 0.70

» Fórmula del riesgo final
risc_final = factor codi * factor edat

Una vez procesados ​​los clientes, debes ser capaz de responder:

* » 1. ¿Qué clientes deben contactar primero en una campaña de retargeting?
* » 2. ¿Qué diferencias existen entre clientes con el mismo comportamiento pero de edades distintas? (Por ejemplo: ¿un cliente joven que abandona la cesta de compra hace 24 h tiene el mismo riesgo que uno de más edad?)
* » 3. ¿Qué clientes presentan tan poco interés que no vale la pena invertir presupuesto?
* » 4. ¿Qué cliente sería lo más rentable de recuperar?
Interpretando el riesgo final y el segmento.

---
## Requisitos técnicos de tu programa

Tendrás que construir un sistema modular utilizando diversas funciones y, sobre todo, sin utilizar if/elif/elsepara realizar la segmentación.

Importante

Para resolver este ejercicio, tendrás que:

 » Desarrollar un sistema modular con las siguientes funciones:

 » Se debe implementar con un diccionario de diccionarios, sin condiciones.

processar_clients# se debe alimentar de las siguientes funciones:
* obtenir_segment_i_factor
* obtenir_factor_edat
* calcular_risc
* mostrar_dataframeomostrar_diccionari
* mostrar_respostes
* main_marketing# orquesta todas las funciones
» Escribir primero el algoritmo paso a paso, indicando claramente qué operaciones realizará cada función.
» Escribir el código de cada función.
 Resultado esperado

Un DataFrame o diccionario final con:

* id_cliente
* número
* edad
* grupo_edad
* codigo_comportamiento
* segmento
* mensaje
* factor_código
* factor_edad
* riesgo_final

Y capacidad para responder a todas las preguntas de marketing pedidas en una función.

 Diccionarios de Mapeo y Funciones de Extracción

---
Para no utilizar sentencias condicionales (`if/elif/else`) en el ejercicio, he implementado una estructura de datos basada en **diccionarios de mapeo**.
Las funciones `obtenir_segment_i_factor` y `obtenir_factor_edat` actúan como simples extractores: reciben una clave (el código o la edad) y devuelven instantáneamente los valores asociados definidos por el departamento de CRM.

In [6]:
MAPEO_COMPORTAMIENTO = {
    '0': {'factor': 0.10, 'segmento': 'Navegador Pasivo', 'mensaje': '¿Buscabas algo especial? Descubre nuestras novedades.'},
    '1': {'factor': 0.25, 'segmento': 'Interés Inicial', 'mensaje': 'Tus favoritos te esperan. ¡Completa tu carrito hoy!'},
    '2': {'factor': 0.50, 'segmento': 'Abandono en Checkout', 'mensaje': '¿Hubo algún problema? Te ayudamos a finalizar tu compra.'},
    '3': {'factor': 0.75, 'segmento': 'Abandono Reciente (<24h)', 'mensaje': '¡No lo pierdas! Tu cesta sigue guardada. Envío gratis si compras ya.'},
    '4': {'factor': 0.90, 'segmento': 'Abandono Crítico (>24h)', 'mensaje': 'Última oportunidad: 10% de descuento para recuperar tu cesta.'}
}

MAPEO_EDAD = {
    '18-25': 1.20,
    '26-40': 1.00,
    '41-60': 0.85,
    '60+':   0.70
}

def obtenir_segment_i_factor(codigo):
    return MAPEO_COMPORTAMIENTO[str(codigo)]

def obtenir_factor_edat(grupo_edad):
    return MAPEO_EDAD[grupo_edad]

### Lógica de Negocio y Procesamiento

---

En esta fase, he construido el motor del programa. La función `calcular_risc` encapsula la fórmula matemática suministrada por negocio, asegurando que el resultado se redondee a 3 decimales para mantener la limpieza visual.
La función `processar_clients` se encarga de la transformación de los datos (*ETL*). Mediante un bucle `for`, recorre la lista original de clientes, invoca a las funciones de extracción creadas en el paso anterior para obtener los factores, calcula el riesgo ponderado y construye una nueva lista de diccionarios con la información enriquecida, lista para ser analizada por el equipo de Marketing.

In [7]:
def calcular_risc(factor_codigo, factor_edad):
    return round(factor_codigo * factor_edad, 3)

def processar_clients(lista_clientes):
    clientes_procesados = []

    for cliente in lista_clientes:
        info_comp = obtenir_segment_i_factor(cliente['codi_comportament'])
        f_edad = obtenir_factor_edat(cliente['grup_edat'])

        f_codigo = info_comp['factor']
        segmento = info_comp['segmento']
        mensaje = info_comp['mensaje']

        riesgo = calcular_risc(f_codigo, f_edad)

        clientes_procesados.append({
            'id_cliente': cliente['id_client'],
            'nombre': cliente['nom'],
            'edad': cliente['edat'],
            'grupo_edad': cliente['grup_edat'],
            'codigo_comportamiento': cliente['codi_comportament'],
            'segmento': segmento,
            'mensaje': mensaje,
            'factor_código': f_codigo,
            'factor_edad': f_edad,
            'riesgo_final': riesgo
        })

    return clientes_procesados

### Función de Salida y Presentación Tabular

---
Primero he diseñado la función `mostrar_dataframe_o_diccionari` utilizando formateo de cadenas de texto avanzado (f-strings con alineación) para los datos crudos dificiles de interpretar. Esto transforma la lista de diccionarios en una tabla visualmente estructurada y escaneable por consola, permitiendo identificar rápidamente los segmentos y los niveles de riesgo de un vistazo.

In [8]:
def mostrar_dataframe_o_diccionari(clientes_procesados):
    print("\n" + "="*145)
    print(f"{'ID':<4} | {'Nombre':<15} | {'Edad':<4} | {'Grupo Edad':<10} | {'Cód':<3} | "
          f"{'Segmento':<24} | {'Factor C.':<9} | {'Factor E.':<9} | {'Riesgo F.':<9} | {'Mensaje Publicitario'}")
    print("-" * 145)

    for c in clientes_procesados:
        print(f"{c['id_cliente']:<4} | {c['nombre']:<15} | {c['edad']:<4} | {c['grupo_edad']:<10} | "
              f"{c['codigo_comportamiento']:<3} | {c['segmento']:<24} | {c['factor_código']:<9} | "
              f"{c['factor_edad']:<9} | {c['riesgo_final']:<9} | {c['mensaje']}")
    print("="*145)

### Dataset Oficial y Orquestación del Programa

---
En la función `mostrar_respostes`, primero ordeno la lista de clientes procesados de mayor a menor riesgo utilizando `sorted()` y una función lambda.
A partir de este dataset ordenado, aplique comprensiones de lista para aislar a los perfiles que necesita:
1. **Prioridad alta:** Filtro los clientes cuyo riesgo es igual o superior a 0.90.
2. **Impacto de la edad:** Demuestro como el multiplicador generacional penaliza (o eleva) el riesgo ante el mismo patrón de abandono.
3. **Poco interés:** Aíslo a aquellos con un riesgo mínimo (<= 0.20) para evitar malgastar el presupuesto publicitario.
4. **Rentabilidad:** Busco a los clientes en los segmentos críticos (2 y 3) que muestran la mayor "intención de compra" bloqueada.

Finalmente, la función `main_marketing` orquesta y ejecuta todo el pipeline de forma secuencial.

In [9]:
def mostrar_respostes(clientes_procesados):
    por_riesgo = sorted(clientes_procesados, key=lambda x: x['riesgo_final'], reverse=True)

    print("\n INFORME ESTRATÉGICO DE MARKETING DIGITAL (RESULTADOS OFICIALES)")
    print("-" * 65)

    alta_prioridad = [c['nombre'] for c in por_riesgo if c['riesgo_final'] >= 0.90]
    print(f"» 1. Qué clientes deben contactar primero en una campaña de retargeting?\n"
          f"   -> {', '.join(alta_prioridad)} (Presentan los riesgos más elevados de la base de datos).")

    print("\n» 2. ¿Qué diferencias existen entre clientes con el mismo comportamiento pero de edades distintas?\n"
          "   -> La edad altera radicalmente la urgencia.")
    print(f"      Ejemplo con tus datos (Código 4 - Abandono Crítico):\n"
          f"      - Laura Serra (18-25) tiene un riesgo final de {por_riesgo[0]['riesgo_final']} (Multiplicador 1.20).\n"
          f"      - Mónica Pérez (41-60) baja a un riesgo de {round(0.90*0.85,3)} (Multiplicador 0.85).")
    print("      Conclusión: A igual comportamiento, el cliente más joven siempre tiene mayor riesgo final.")

    poco_interes = [c['nombre'] for c in por_riesgo if c['riesgo_final'] <= 0.20]
    print(f"\n» 3. ¿Qué clientes presentan tan poco interés que no vale la pena invertir presupuesto?\n"
          f"   -> {', '.join(poco_interes)} (Navegaron o hicieron carrito inicial pero su riesgo ponderado es mínimo).")

    rentables = [c['nombre'] for c in por_riesgo if c['codigo_comportamiento'] in [2, 3]][:2]
    print(f"\n» 4. ¿Qué cliente sería lo más rentable de recuperar?\n"
          f"   -> {', '.join(rentables)}\n"
          f"   -> Justificación analítica: Estuvieron en la pasarela de pago o abandonaron hace menos de 24h.\n"
          f"      Su 'intención de compra' es altísima; recuperarlos cuesta menos presupuesto y genera conversión inmediata.")

clients = [
    {"id_client": 1, "nom": "Laura Serra", "edat": 23, "grup_edat": "18-25", "codi_comportament": 4},
    {"id_client": 2, "nom": "Marc Vidal", "edat": 35, "grup_edat": "26-40", "codi_comportament": 3},
    {"id_client": 3, "nom": "Ana López", "edat": 29, "grup_edat": "26-40", "codi_comportament": 2},
    {"id_client": 4, "nom": "Joan Riera", "edat": 19, "grup_edat": "18-25", "codi_comportament": 2},
    {"id_client": 5, "nom": "Mónica Pérez", "edat": 47, "grup_edat": "41-60", "codi_comportament": 4},
    {"id_client": 6, "nom": "Luis García", "edat": 52, "grup_edat": "41-60", "codi_comportament": 1},
    {"id_client": 7, "nom": "Pilar Sánchez", "edat": 61, "grup_edat": "60+", "codi_comportament": 3},
    {"id_client": 8, "nom": "Eva Martín", "edat": 38, "grup_edat": "26-40", "codi_comportament": 0},
    {"id_client": 9, "nom": "Diego Romero", "edat": 24, "grup_edat": "18-25", "codi_comportament": 1},
    {"id_client": 10, "nom": "Núria Costa", "edat": 33, "grup_edat": "26-40", "codi_comportament": 4}
]

def main_marketing():
    datos_enriquecidos = processar_clients(clients)
    mostrar_dataframe_o_diccionari(datos_enriquecidos)
    mostrar_respostes(datos_enriquecidos)

main_marketing()


ID   | Nombre          | Edad | Grupo Edad | Cód | Segmento                 | Factor C. | Factor E. | Riesgo F. | Mensaje Publicitario
-------------------------------------------------------------------------------------------------------------------------------------------------
1    | Laura Serra     | 23   | 18-25      | 4   | Abandono Crítico (>24h)  | 0.9       | 1.2       | 1.08      | Última oportunidad: 10% de descuento para recuperar tu cesta.
2    | Marc Vidal      | 35   | 26-40      | 3   | Abandono Reciente (<24h) | 0.75      | 1.0       | 0.75      | ¡No lo pierdas! Tu cesta sigue guardada. Envío gratis si compras ya.
3    | Ana López       | 29   | 26-40      | 2   | Abandono en Checkout     | 0.5       | 1.0       | 0.5       | ¿Hubo algún problema? Te ayudamos a finalizar tu compra.
4    | Joan Riera      | 19   | 18-25      | 2   | Abandono en Checkout     | 0.5       | 1.2       | 0.6       | ¿Hubo algún problema? Te ayudamos a finalizar tu compra.
5    | Mónica Pér

## Nivel 2
###Ejercicio 1: Gestión de riesgo y priorización de pacientes

Trabajas como analista de datos en un centro médico especializado . Varios pacientes han enviado sus síntomas a través de un formulario online, pero los datos llegan en un formato desordenado y poco estructurado , y el equipo médico necesita que los organices para priorizar el orden de atención según la gravedad.

A continuación tienes una lista con información real procedente del sistema:

                  

pacients = [

    "Maria|45|tos, Fiebre Alta, dolor_pecho",
    "Luis|33|dolor_cabeza,faTiga",
    "Sara|67|fiebre alta, dificultad RESPIRAR, tos",
    "Jordi|52|fatiga , tos",
    "Anna|29|fiebre alta,dolor_cabeza"
]

                
El departamento médico también te ha proporcionado el siguiente diccionario de niveles de riesgo por síntoma:

                  

nivells_risc = {

    "fiebre_alta": 3,
    "dificultad_respirar": 5,
    "dolor_pecho": 4,
    "tos": 1,
    "fatiga": 1,
    "dolor_cabeza": 1
}

                
Objetivos

El objetivo es transformar esos datos en información útil para los médicos. A partir de los registros en bruto, debes:

» Convertir cada string en una estructura limpia (diccionario o similar).

» Separar y normalizar la lista de síntomas:
* minúsculas
* eliminar espacios
* dividir correctamente por comas
» Asignar un riesgo a cada síntoma utilizando el diccionario nivells_risc.

» Calcular el riesgo total de cada paciente sumando los valores.

» Ordenar a los pacientes de mayor a menor riesgo para determinar la prioridad de atención.








### 1. Funciones de Limpieza y Normalización de Texto

---
Los datos crudos de los pacientes presentaban un alto nivel de errores estructurales (espacios en blanco, mezcla de mayúsculas/minúsculas y formatos inconsistentes). Para poder cruzar esta información con el diccionario médico, hice un proceso de limpieza:
1. Utilicé la función `split("|")` para dividir el bloque de texto principal en sus tres componentes lógicos (nombre, edad y síntomas).
2. La función `normalizar_sintoma` es vital para el éxito del cruce de datos. Encadeno los métodos `.strip()` (para eliminar espacios fantasma en los extremos), `.lower()` (para estandarizar todo a minúsculas) y `.replace(" ", "_")` para convertir, por ejemplo, "Fiebre Alta" exactamente en "fiebre_alta", coincidiendo al 100% con las claves del diccionario de riesgo.

In [10]:
nivells_risc = {
    "fiebre_alta": 3,
    "dificultad_respirar": 5,
    "dolor_pecho": 4,
    "tos": 1,
    "fatiga": 1,
    "dolor_cabeza": 1
}

def normalizar_sintoma(sintoma_bruto):

    texto_limpio = sintoma_bruto.strip().lower()
    return texto_limpio.replace(" ", "_")


def parsear_registro_paciente(registro_string):
    partes = registro_string.split("|")
    nombre = partes[0].strip()
    edad = int(partes[1].strip())
    sintomas_lista = partes[2].split(",")

    return nombre, edad, sintomas_lista

### 2. Lógica de Negocio y Cálculo del Riesgo Clínico

---
En esta etapa diseñé el motor de transformación de datos (ETL).
* Para calcular la gravedad (`calcular_riesgo_paciente`), implementé el método `.get(sintoma, 0)`. Esta decisión técnica garantiza que si en el futuro un paciente introduce un síntoma con error ortográfico o no contemplado en el diccionario médico, el programa no colapse con un error `KeyError`, sino que sume 0 a la puntuación de ese síntoma y continúe operando.
* La función `procesar_historial_medico` orquesta la limpieza: recorre el registro, utiliza comprensiones de lista para limpiar todos los síntomas de forma eficiente, calcula el riesgo total y empaqueta al paciente en un diccionario estructurado.

In [11]:
def calcular_riesgo_paciente(sintomas_normalizados):
    riesgo_total = 0
    for sintoma in sintomas_normalizados:
        riesgo_total += nivells_risc.get(sintoma, 0)
    return riesgo_total


def procesar_historial_medico(lista_pacientes_bruto):
    pacientes_procesados = []

    for registro in lista_pacientes_bruto:
        nombre, edad, sintomas_brutos = parsear_registro_paciente(registro)

        sintomas_limpios = [normalizar_sintoma(s) for s in sintomas_brutos]

        total_riesgo = calcular_riesgo_paciente(sintomas_limpios)

        pacientes_procesados.append({
            'nombre': nombre,
            'edad': edad,
            'sintomas': sintomas_limpios,
            'riesgo_total': total_riesgo
        })

    return pacientes_procesados

### 3. Función de Salida y Presentación del Triaje

---
Diseñé una salida en formato de tabla. Utilicé la función `sorted()` indicando que el criterio principal (`key`) fuera el `riesgo_total` matemático generado anteriormente, y forzando un orden descendente (`reverse=True`) para que los casos más graves (Prioridad 1) aparezcan arriba. Finalmente, utilicé f-strings con espaciado definido para crear columnas tabulares limpias y fáciles de escanear visualmente por el equipo médico.

In [12]:
def generar_informe_prioridad(pacientes_limpios):
    print("\n" + "="*85)
    print("      SISTEMA DE TRIAJE Y PRIORIZACIÓN DE PACIENTES - EMERGENCIAS MÉDICAS")
    print("="*85)

    pacientes_ordenados = sorted(pacientes_limpios, key=lambda x: x['riesgo_total'], reverse=True)

    print(f"{'Orden':<5} | {'Paciente':<12} | {'Edad':<4} | {'Riesgo Total':<12} | {'Síntomas Presentados'}")
    print("-" * 85)

    for posicion, paciente in enumerate(pacientes_ordenados, start=1):
        sintomas_texto = ", ".join(paciente['sintomas'])

        print(f"#{posicion:<4} | {paciente['nombre']:<12} | {paciente['edad']:<4} | "
              f"{paciente['riesgo_total']:<12} | {sintomas_texto}")
    print("="*85)

### 4. Dataset Oficial y Ejecución del Orquestador

---
Para mantener el código limpio y modular, mantengo el dataset original separado de la lógica y utilizo una función contenedora o "wrapper" (`main_centro_medico()`). Esta función actúa como el punto de entrada único de la aplicación, ejecutando el flujo completo desde la ingesta de los datos brutos hasta la impresión del informe de triaje.

In [13]:
pacients = [
    "Maria|45|tos, Fiebre Alta, dolor_pecho",
    "Luis|33|dolor_cabeza,faTiga",
    "Sara|67|fiebre alta, dificultad RESPIRAR, tos",
    "Jordi|52|fatiga , tos",
    "Anna|29|fiebre alta,dolor_cabeza"
]


def main_centro_medico():

    base_datos_pacientes = procesar_historial_medico(pacients)

    generar_informe_prioridad(base_datos_pacientes)


main_centro_medico()


      SISTEMA DE TRIAJE Y PRIORIZACIÓN DE PACIENTES - EMERGENCIAS MÉDICAS
Orden | Paciente     | Edad | Riesgo Total | Síntomas Presentados
-------------------------------------------------------------------------------------
#1    | Sara         | 67   | 9            | fiebre_alta, dificultad_respirar, tos
#2    | Maria        | 45   | 8            | tos, fiebre_alta, dolor_pecho
#3    | Anna         | 29   | 4            | fiebre_alta, dolor_cabeza
#4    | Luis         | 33   | 2            | dolor_cabeza, fatiga
#5    | Jordi        | 52   | 2            | fatiga, tos


# Ejercicio 2: Clasificación de clientes de seguros

Trabajas como analista de datos en una compañía aseguradora.

La empresa quiere calcular el riesgo de los clientes, el precio estimado de la póliza y la probabilidad de fraude utilizando un código de comportamiento que ha sido generado por un modelo interno.



Tienes la siguiente lista de clientes:

                  

clients = [

    {"nom": "Ana", "edat": 42, "codi": 0},
    {"nom": "Carlos", "edat": 30, "codi": 2},
    {"nom": "Isabel", "edat": 55, "codi": 3},
    {"nom": "Jorge", "edat": 40, "codi": 1},
    {"nom": "Marta", "edat": 28, "codi": 0}
]

                  

                
Y este diccionario con las categorías del sistema:

                  

nivells_seguro = {

    0: {"categoria": "Baix risc", "preu_base": 120, "fraude": 0.01},
    1: {"categoria": "Risc mitjà", "preu_base": 200, "fraude": 0.05},
    2: {"categoria": "Risc alt", "preu_base": 350, "fraude": 0.15},
    3: {"categoria": "Risc crític", "preu_base": 500, "fraude": 0.30}
}
                  

                
El precio final debe calcularse con la fórmula:


preu_final = preu_base * (1 + edat/100)

---

## 🎯 Objetivos

» Clasificar cada cliente utilizando niveles_seguro ( sense if/elif).


» Añadir:

* La categoría

* El precio base

* La probabilidad de fraude

» Calcular el precio final de la póliza con la fórmula dada.

» Crear un DataFrame o diccionario con la información final.

» Ordenar a los clientes:

* Por probabilidad de fraude (descendente).

* Por precio final.

* Importante

Al igual que en el Ejercicio 1, en este nivel eres tú quien debe definir las funciones necesarias. Se recomienda mantener el código modular, donde:

» Cada función resuelve un paso del proceso.

» La función principal delega todas las tareas.

✅ Resultado esperado

Un DataFrame o un diccionario con:

* Nombre
* Edad
* Categoría
* Precio_base
* Precio_final
* Probabilidad_fraude
* Un apartado de análisis donde se indique:
* Qué cliente tiene más probabilidad de fraude
* ¿Cuál tiene el precio final más alto
* Quien sería prioritario revisar manualmente

### 1. Funciones de Mapeo y Cálculo de Póliza

---
Para cumplir con el requerimiento estricto de clasificar a los clientes sin utilizar sentencias condicionales (`if/elif/else`), he implementado un sistema de búsqueda utilizando el diccionario maestro `nivells_seguro`.
Por otro lado, he aislado el cálculo financiero en la función `calcular_precio_final`, la cual aplica la fórmula matemática de ponderación por edad exigida por el negocio. He añadido un redondeo a dos decimales para mantener la coherencia y legibilidad al tratarse de importes monetarios.

In [14]:
nivells_seguro = {
    0: {"categoria": "Baix risc", "preu_base": 120, "fraude": 0.01},
    1: {"categoria": "Risc mitjà", "preu_base": 200, "fraude": 0.05},
    2: {"categoria": "Risc alt", "preu_base": 350, "fraude": 0.15},
    3: {"categoria": "Risc crític", "preu_base": 500, "fraude": 0.30}
}

def calcular_precio_final(precio_base, edad):
    return round(precio_base * (1 + edad / 100), 2)

### 2. Lógica de Negocio y Enriquecimiento del Dataset

---
En esta etapa he diseñado el proceso de transformación (ETL). La función `processar_cartera_clientes` itera sobre la base de datos original. Por cada cliente, utiliza su código de comportamiento como "llave" para extraer instantáneamente su categoría, precio base y probabilidad de fraude del diccionario maestro. Una vez obtenidos estos factores, invoca a la función de cálculo financiero y consolida todas las métricas en un nuevo diccionario enriquecido, preservando la integridad de los datos originales.

In [15]:
def processar_cartera_clientes(lista_clientes):
    clientes_procesados = []

    for cliente in lista_clientes:
        datos_seguro = nivells_seguro[cliente['codi']]

        p_base = datos_seguro['preu_base']
        edad = cliente['edat']

        p_final = calcular_precio_final(p_base, edad)

        clientes_procesados.append({
            'nombre': cliente['nom'],
            'edad': edad,
            'categoria': datos_seguro['categoria'],
            'precio_base': p_base,
            'precio_final': p_final,
            'probabilidad_fraude': datos_seguro['fraude']
        })

    return clientes_procesados

### 3. Funciones de Salida y Reporte de Análisis de Riesgos

---
Para que la información sea procesable, he estructurado la salida en dos bloques.
1. En `mostrar_tabla_seguros`, implemento el ordenamiento doble exigido utilizando `sorted()`. Al pasar una tupla en la función lambda `(x['probabilidad_fraude'], x['precio_final'])` y forzar el orden descendente, garantizo que los perfiles más peligrosos y de mayor valor queden en la parte superior del reporte tabular.
2. En `generar_analisis_negocio`, automatizo las respuestas a las preguntas de auditoría. Dado que la lista ya llega ordenada por riesgo, extraigo al cliente con mayor probabilidad de fraude simplemente tomando la posición `[0]`. Para encontrar la póliza más cara de forma segura, utilizo la función integrada `max()` aplicada específicamente sobre el campo de precio final.

In [16]:
def mostrar_tabla_seguros(clientes_procesados):
    print("\n" + "="*95)
    print("         INFORME DE CONTROL DE RIESGOS Y PÓLIZAS - COMPAÑÍA ASEGURADORA")
    print("="*95)

    clientes_ordenados = sorted(
        clientes_procesados,
        key=lambda x: (x['probabilidad_fraude'], x['precio_final']),
        reverse=True
    )

    print(f"{'Nombre':<12} | {'Edad':<4} | {'Categoría Riesgo':<15} | {'Precio Base':<12} | {'Precio Final':<13} | {'Fraude'}")
    print("-" * 95)

    for c in clientes_ordenados:
        fraude_porcentaje = f"{int(c['probabilidad_fraude'] * 100)}%"

        print(f"{c['nombre']:<12} | {c['edad']:<4} | {c['categoria']:<15} | "
              f"${c['precio_base']:<11} | ${c['precio_final']:<12} | {fraude_porcentaje}")
    print("="*95)
    return clientes_ordenados


def generar_analisis_negocio(clientes_ordenados):
    print("\n🔍 APARTADO DE ANÁLISIS Y AUDITORÍA")
    print("-" * 40)

    critico_fraude = clientes_ordenados[0]

    mas_caro = max(clientes_ordenados, key=lambda x: x['precio_final'])

    print(f"» Cliente con más probabilidad de fraude:\n"
          f"  -> {critico_fraude['nombre']} ({int(critico_fraude['probabilidad_fraude']*100)}% de probabilidad).\n")

    print(f"» Cliente con el precio final más alto:\n"
          f"  -> {mas_caro['nombre']} (${mas_caro['precio_final']} debido a su edad de {mas_caro['edad']} años).\n")

    print(f"» Prioritario revisar manualmente:\n"
          f"  -> {critico_fraude['nombre']} (Posee una categoría de '{critico_fraude['categoria']}' con un {int(critico_fraude['probabilidad_fraude']*100)}% de riesgo de fraude) .")

### 4. Dataset Inicial y Orquestación General del Sistema

---
Para garantizar la escalabilidad y limpieza del cuaderno, he separado el listado de datos (`clients`) de la lógica del programa. Finalmente, he creado la función orquestadora `main_aseguradora()`, la cual delega y ejecuta secuencialmente todas las fases del proceso: enriquecimiento, visualización tabular y extracción de *insights* de negocio.

In [17]:
clients = [
    {"nom": "Ana", "edat": 42, "codi": 0},
    {"nom": "Carlos", "edat": 30, "codi": 2},
    {"nom": "Isabel", "edat": 55, "codi": 3},
    {"nom": "Jorge", "edat": 40, "codi": 1},
    {"nom": "Marta", "edat": 28, "codi": 0}
]


def main_aseguradora():
    cartera_enriquecida = processar_cartera_clientes(clients)

    cartera_ordenada = mostrar_tabla_seguros(cartera_enriquecida)

    generar_analisis_negocio(cartera_ordenada)


main_aseguradora()


         INFORME DE CONTROL DE RIESGOS Y PÓLIZAS - COMPAÑÍA ASEGURADORA
Nombre       | Edad | Categoría Riesgo | Precio Base  | Precio Final  | Fraude
-----------------------------------------------------------------------------------------------
Isabel       | 55   | Risc crític     | $500         | $775.0        | 30%
Carlos       | 30   | Risc alt        | $350         | $455.0        | 15%
Jorge        | 40   | Risc mitjà      | $200         | $280.0        | 5%
Ana          | 42   | Baix risc       | $120         | $170.4        | 1%
Marta        | 28   | Baix risc       | $120         | $153.6        | 1%

🔍 APARTADO DE ANÁLISIS Y AUDITORÍA
----------------------------------------
» Cliente con más probabilidad de fraude:
  -> Isabel (30% de probabilidad).

» Cliente con el precio final más alto:
  -> Isabel ($775.0 debido a su edad de 55 años).

» Prioritario revisar manualmente:
  -> Isabel (Posee una categoría de 'Risc crític' con un 30% de riesgo de fraude) .


# Nivel 3

##Ejercicio 1: Generador de contraseñas seguras

Eres analista de datos en el departamento de Ciberseguridad de una empresa tecnológica. Se ha detectado que la mayoría de trabajadores siguen utilizando contraseñas muy débiles como “Asdf1234”, fechas de cumpleaños o combinaciones fáciles de adivinar. Por este motivo, el responsable del departamento te ha pedido desarrollar un generador de contraseñas seguras , totalmente parametrizable y modular.

###Requisitos del generador

Tienes que crear una función principal y todas las subfunciones necesarias para:

» Generar contraseñas de longitud variable.

» Decidir si deben incluir:

* Mayúsculas
* Minúsculas
* Números
* Signos especiales

» Asegurarte de que al menos aparece un carácter de cada tipo seleccionado .

» Aleatorizar completamente la contraseña.

» Devolver la contraseña generada.

###Nota importante

Explora el funcionamiento del módulo random de la librería icono enlacenumpy

La función principal del generador debe tener valores por defecto, exactamente los siguientes:

* majuscules=True
* minuscules=True
* numeros=True
* signes=False

Estos parámetros son obligatorios y forman parte del diseño del sistema.

### 1. Constantes y Funciones Modulares de Construcción y Selección Aleatoria

---
Para desarrollar un generador de contraseñas, mi primera decisión fue importar la librería `string` para definir exactamente los caracteres válidos (mayúsculas, minúsculas, números y signos) y la librería `numpy` para el motor de aleatoriedad.

La función `construir_pool_y_obligatorios` aplica una lógica estricta de cumplimiento: itera sobre las opciones activadas por el usuario y, por cada una, añade todo su set de caracteres al *pool* general y extrae obligatoriamente un carácter al azar (usando `np.random.choice`). Esto garantiza matemáticamente que la contraseña final contenga, como mínimo, un carácter de cada tipo exigido.
Posteriormente, la función `completar_y_mezclar_contrasenia` rellena la longitud restante seleccionando del *pool* general y aplica `np.random.shuffle` para destruir cualquier patrón, maximizando la seguridad de la clave.

In [18]:
import numpy as np
import string

CHARS_MAYUSCULAS = string.ascii_uppercase
CHARS_MINUSCULAS = string.ascii_lowercase
CHARS_NUMEROS = string.digits
CHARS_SIGNOS = string.punctuation

def construir_pool_y_obligatorios(majuscules, minuscules, numeros, signes):
    pool_total = []
    obligatorios = []

    configuraciones = [
        (majuscules, CHARS_MAYUSCULAS),
        (minuscules, CHARS_MINUSCULAS),
        (numeros, CHARS_NUMEROS),
        (signes, CHARS_SIGNOS)
    ]

    for activado, caracteres in configuraciones:
        if activado:
            lista_caracteres = list(caracteres)
            pool_total.extend(lista_caracteres)

            caracter_seguro = np.random.choice(lista_caracteres)
            obligatorios.append(caracter_seguro)

    return pool_total, obligatorios


def completar_y_mezclar_contrasenia(pool_total, obligatorios, longitud_restante):
    caracteres_restantes = np.random.choice(pool_total, size=longitud_restante)

    contrasenia_lista = obligatorios + list(caracteres_restantes)

    np.random.shuffle(contrasenia_lista)

    return "".join(contrasenia_lista)

### 2. Función Principal del Generador (Valores por Defecto Obligatorios)

---
He diseñado la función `generar_contrasenia` como el núcleo orquestador del sistema, inyectando exactamente los parámetros por defecto obligatorios que solicitaba el departamento de Ciberseguridad (`majuscules=True`, `minuscules=True`, `numeros=True`, `signes=False`).

Además, he implementado programación defensiva (control de errores). El código verifica lógicamente dos escenarios de fallo humano:
1. Que el usuario no desactive todos los tipos de caracteres (dejando el *pool* vacío).
2. Que la longitud solicitada no sea inferior a la cantidad de reglas obligatorias seleccionadas (por ejemplo, pedir una contraseña de 2 caracteres pero exigir mayúsculas, minúsculas y números).
Si pasa los controles, delega la tarea a las subfunciones de generación.

In [19]:
def generar_contrasenia(longitud=12, majuscules=True, minuscules=True, numeros=True, signes=False):
    pool, obligatorios = construir_pool_y_obligatorios(majuscules, minuscules, numeros, signes)

    if not pool:
        return "Error: Debes seleccionar al menos un tipo de carácter."

    if longitud < len(obligatorios):
        return f"Error: Longitud insuficiente. Para tus opciones necesitas mínimo {len(obligatorios)} caracteres."

    longitud_restante = longitud - len(obligatorios)

    contrasenia_final = completar_y_mezclar_contrasenia(pool, obligatorios, longitud_restante)

    return contrasenia_final

### 3. Pruebas de Auditoría de Ciberseguridad

---
Para validar la funcionalidad del sistema antes de su despliegue, he creado la función `main_ciberseguridad()`. Esta función actúa como una batería de pruebas automatizada, simulando diferentes escenarios de negocio: desde la creación de contraseñas con la política por defecto, hasta la generación de claves críticas (16 caracteres con signos especiales) y la simulación de PINs restringidos. Esto demuestra al equipo directivo la flexibilidad absoluta y el correcto funcionamiento del diseño modular.

In [20]:
def main_ciberseguridad():
    print("==========================================================")
    print("      AUDITORÍA DE GENERACIÓN DE CONTRASEÑAS SEGURAS")
    print("==========================================================\n")

    print("» 1. Configuración por defecto (Longitud 12):")
    print(f"   -> Password: {generar_contrasenia()}")
    print(f"   -> Password: {generar_contrasenia()}\n")

    print("» 2. Configuración Máxima Seguridad (Longitud 16 + Signos especiales):")
    print(f"   -> Password: {generar_contrasenia(longitud=16, signes=True)}\n")

    print("» 3. Configuración Restringida (Longitud 8, solo Números y Signos):")
    print(f"   -> Password: {generar_contrasenia(longitud=8, majuscules=False, minuscules=False, numeros=True, signes=True)}\n")

    print("==========================================================")

main_ciberseguridad()

      AUDITORÍA DE GENERACIÓN DE CONTRASEÑAS SEGURAS

» 1. Configuración por defecto (Longitud 12):
   -> Password: Mx068ueI2Hku
   -> Password: LISpAbrvBP3l

» 2. Configuración Máxima Seguridad (Longitud 16 + Signos especiales):
   -> Password: #])Vc(8Ya@n'CbM"

» 3. Configuración Restringida (Longitud 8, solo Números y Signos):
   -> Password: %9#@9)1@



## Ejercicio 2: Procesamiento automático de datos deportivos

Ahora formas parte de un equipo de analítica en una organización deportiva que gestiona datos de la liga catalana. Tu compañera de trabajo desea automatizar la lectura de su archivo histórico de resultados y extraer las estadísticas más importantes. Te ha pedido hacer un pequeño sistema en Python que procese el archivo "historic partits.txt"

### Objetivos

Mediante un código modular, claro y documentado , tu programa debe calcular:

» 1.Qué equipo es el más goleador.

» 2.Cuál es el más goleado.

» 3.El número total de goles que ha marcado cada equipo.

» 4.La clasificación final ordenada por puntos donde cada victoria suma 3 pts, cada empate 1 pts y cada derrota 0 pts.

En este nivel no se da ninguna función. Eres tú quien debe diseñar un flujo completo. El código debe reflejar una buena organización interna.

### 1. Funciones de Inicialización y Parsing de Partidos

---
El primer gran reto de este ejercicio era lidiar con un archivo de texto en bruto y convertirlo en datos estructurados.
* Diseñé la función `parsear_linea_partido` aplicando expresiones regulares (`re.sub`) y métodos de limpieza (`strip` y `split`) para extraer limpiamente el equipo local, los goles y el visitante. He envuelto esta lógica en un bloque `try-except` como medida de programación defensiva, asegurando que si una línea del archivo está corrupta o vacía, el programa la ignore sin detener la ejecución.
* Además, creé la función de soporte `asegurar_equipo`. Su objetivo es inicializar dinámicamente cada club nuevo que aparece en el texto, creando un diccionario interno con todos sus contadores a cero. Esto previene errores de tipo `KeyError` al momento de sumar los goles o los puntos.

In [1]:
import re

def asegurar_equipo(diccionario, equipo):
    if equipo not in diccionario:
        diccionario[equipo] = {
            "gols_favor": 0,
            "gols_contra": 0,
            "guanyats": 0,
            "empatats": 0,
            "perduts": 0,
            "punts": 0
        }

def parsear_linea_partido(linea):
    linea_limpia = re.sub(r'"', "", linea).strip()

    if not linea_limpia or "-" not in linea_limpia:
        return None, None, None, None

    try:
        partes = [p.strip() for p in linea_limpia.split('\t') if p.strip()]
        if len(partes) < 3:
            return None, None, None, None

        equipo_local = partes[0]
        resultado = partes[1]
        equipo_visitante = partes[2]

        goles_local, goles_visitante = map(int, resultado.split('-'))
        return equipo_local, goles_local, goles_visitante, equipo_visitante
    except Exception:
        return None, None, None, None

### 2. Lógica de Procesamiento de la Competición

---
En esta fase he construido el motor algorítmico del programa.
La función `actualizar_marcadores_y_puntos` aplica estrictamente las reglas de negocio de la liga: suma los goles correspondientes y evalúa con condicionales si el resultado implica una victoria (3 puntos), empate (1 punto) o derrota (0 puntos). He utilizado un parámetro booleano (`es_local`) para reutilizar la misma función tanto para el equipo de casa como para el visitante, manteniendo el código modular (principio DRY).
Finalmente, `procesar_archivo_liga` actúa como la función de ingesta general (ETL), recorriendo el archivo línea a línea y delegando las tareas de extracción y actualización a las subfunciones previas.

In [2]:
def actualizar_marcadores_y_puntos(stats, g_loc, g_vis, es_local):
    if es_local:
        stats["gols_favor"] += g_loc
        stats["gols_contra"] += g_vis
        if g_loc > g_vis:
            stats["guanyats"] += 1
            stats["punts"] += 3
        elif g_loc == g_vis:
            stats["empatats"] += 1
            stats["punts"] += 1
        else:
            stats["perduts"] += 1
    else:
        stats["gols_favor"] += g_vis
        stats["gols_contra"] += g_loc
        if g_vis > g_loc:
            stats["guanyats"] += 1
            stats["punts"] += 3
        elif g_vis == g_loc:
            stats["empatats"] += 1
            stats["punts"] += 1
        else:
            stats["perduts"] += 1

def procesar_archivo_liga(contenido_texto):
    equipos_master = {}
    lineas = contenido_texto.strip().split('\n')

    for linea in lineas:
        eq_local, g_loc, g_vis, eq_vis = parsear_linea_partido(linea)
        if eq_local is None:
            continue

        asegurar_equipo(equipos_master, eq_local)
        asegurar_equipo(equipos_master, eq_vis)

        actualizar_marcadores_y_puntos(equipos_master[eq_local], g_loc, g_vis, es_local=True)
        actualizar_marcadores_y_puntos(equipos_master[eq_vis], g_loc, g_vis, es_local=False)

    return equipos_master

### 3. Funciones Analíticas y Renderizado de Clasificación

---
Los datos procesados necesitan convertirse en *insights* de negocio.
Para responder a las preguntas sobre capacidad goleadora y debilidad defensiva, implementé la función `obtener_extremos_goleadores`, utilizando `max()` iterando directamente sobre los valores del diccionario.
Para la tabla final (`mostrar_tabla_clasificacion`), utilicé la función `sorted()`. Tomé una decisión analítica avanzada: la clave de ordenamiento (`key`) no solo evalúa los puntos totales, sino que, en caso de empate, resuelve la posición utilizando el diferencial de goles general (goles a favor menos goles en contra). Posteriormente, le di formato con f-strings para obtener una tabla perfectamente alineada.

In [3]:
def obtener_extremos_goleadores(equipos_master):
    mas_goleador = max(equipos_master.items(), key=lambda x: x[1]["gols_favor"])
    mas_goleado = max(equipos_master.items(), key=lambda x: x[1]["gols_contra"])
    return mas_goleador, mas_goleado


def mostrar_tabla_clasificacion(equipos_master):
    print("\n" + "="*90)
    print("         CLASIFICACIÓN FINAL DE LA LIGA CATALANA - REPORTE DE ANALÍTICA")
    print("="*90)

    clasificacion_ordenada = sorted(
        equipos_master.items(),
        key=lambda x: (x[1]["punts"], x[1]["gols_favor"] - x[1]["gols_contra"]),
        reverse=True
    )

    print(f"{'equip':<20} | {'gols_favor':<10} | {'gols_contra':<11} | "
          f"{'guanyats':<8} | {'empatats':<8} | {'perduts':<7} | {'punts'}")
    print("-" * 90)

    for equip, stats in clasificacion_ordenada:
        print(f"{equip:<20} | {stats['gols_favor']:<10} | {stats['gols_contra']:<11} | "
              f"{stats['guanyats']:<8} | {stats['empatats']:<8} | {stats['perduts']:<7} | {stats['punts']}")
    print("="*90)

### 4. Orquestación del Pipeline y Ejecución con Datos Reales

---
Mantengo el código altamente modular separando el inmenso bloque de datos (`historico_partidos_txt`) de las funciones de procesamiento. La función `main_analitica_deportiva` actúa como el panel de control de todo el sistema, invocando la carga, gestionando los cálculos intermedios e imprimiendo los resultados estratégicos definitivos para el departamento de análisis deportivo.

In [5]:
historico_partidos_txt = """
Manlleu	0-1	Granollers
Vilafranca	4-5	Terrassa
Olot	5-0	Granollers
Vilafranca	5-5	Manlleu
Cornellà	5-1	Prat
Cerdanyola	0-5	Vilafranca
Sant Andreu	2-3	Cornellà
Vilafranca	5-4	Olot
FC Barcelona	4-4	Badalona
Granollers	1-5	Sabadell
Granollers	0-5	Europa
Prat	5-2	Llagostera
Llagostera	2-5	Vilafranca
Sant Andreu	1-4	Figueres
Figueres	1-2	Cerdanyola
Granollers	0-4	Cornellà
Figueres	2-3	RCD Espanyol
Llagostera	2-5	Figueres
Cornellà	3-1	Reus Deportiu
Europa	5-1	Granollers
Terrassa	0-4	Europa
Olot	1-2	Sant Andreu
Vilafranca	5-2	Europa
Vilafranca	1-5	Reus Deportiu
Reus Deportiu	4-4	Nàstic de Tarragona
Prat	3-0	Lleida Esportiu
Sant Andreu	1-2	Cerdanyola
Granollers	2-0	Prat
Vilafranca	5-5	Terrassa
Girona FC	3-0	Nàstic de Tarragona
Cerdanyola	2-4	Olot
Prat	3-2	Nàstic de Tarragona
Figueres	4-4	Terrassa
Badalona	4-3	Sant Andreu
Badalona	3-4	Manlleu
Terrassa	4-0	Nàstic de Tarragona
Reus Deportiu	1-0	Llagostera
Manlleu	2-3	Badalona
Granollers	2-3	Nàstic de Tarragona
Figueres	4-1	Lleida Esportiu
Olot	1-1	FC Barcelona
Manlleu	2-5	Girona FC
Figueres	5-5	Cornellà
Llagostera	3-1	Figueres
Cerdanyola	5-1	Manlleu
Olot	4-0	FC Barcelona
Badalona	4-2	FC Barcelona
Sabadell	5-1	Lleida Esportiu
Girona FC	0-3	RCD Espanyol
Terrassa	0-4	Girona FC
Manlleu	2-1	Nàstic de Tarragona
FC Barcelona	5-2	Terrassa
FC Barcelona	5-3	RCD Espanyol
Lleida Esportiu	0-2	Sabadell
Vilafranca	4-2	Badalona
Terrassa	1-2	Cornellà
Sabadell	3-2	Olot
Figueres	2-5	Granollers
Vilafranca	1-2	RCD Espanyol
Cerdanyola	0-0	Manlleu
Vilafranca	3-4	Lleida Esportiu
Vilafranca	4-0	Cerdanyola
Olot	2-3	Badalona
Prat	5-2	Cerdanyola
Cornellà	3-5	Manlleu
Cornellà	0-3	Lleida Esportiu
Badalona	5-4	Lleida Esportiu
Llagostera	2-0	Terrassa
Badalona	3-3	Europa
Llagostera	0-4	Europa
Vilafranca	4-0	Prat
Cerdanyola	5-3	Vilafranca
Girona FC	3-4	Lleida Esportiu
Girona FC	1-0	Llagostera
Cerdanyola	4-1	Girona FC
Cerdanyola	2-5	Lleida Esportiu
Manlleu	5-0	Girona FC
Granollers	4-5	RCD Espanyol
Badalona	2-4	Olot
Lleida Esportiu	2-3	Girona FC
Figueres	3-1	Badalona
Terrassa	4-5	Cornellà
RCD Espanyol	1-2	Europa
Europa	0-1	Vilafranca
Nàstic de Tarragona	2-5	Figueres
Sant Andreu	1-5	Manlleu
Europa	5-3	FC Barcelona
Sant Andreu	1-3	Lleida Esportiu
Cerdanyola	2-4	Badalona
Sant Andreu	4-0	Terrassa
Sabadell	4-3	Europa
Badalona	3-1	Nàstic de Tarragona
Granollers	4-1	Vilafranca
Reus Deportiu	4-1	Vilafranca
Prat	1-5	Granollers
RCD Espanyol	0-5	Nàstic de Tarragona
Badalona	0-2	Sabadell
Girona FC	3-3	Badalona
Terrassa	1-0	Sabadell
Lleida Esportiu	2-5	Figueres
Olot	2-2	Figueres
RCD Espanyol	0-4	Llagostera
Badalona	5-0	Figueres
Prat	1-2	Olot
Cerdanyola	0-2	Lleida Esportiu
FC Barcelona	2-5	Figueres
Terrassa	5-5	Badalona
Nàstic de Tarragona	5-4	Badalona
Nàstic de Tarragona	0-4	Terrassa
Sant Andreu	3-4	Nàstic de Tarragona
Cornellà	3-3	Reus Deportiu
Cerdanyola	4-1	Manlleu
Sabadell	3-3	Nàstic de Tarragona
Reus Deportiu	4-5	Nàstic de Tarragona
Cerdanyola	2-0	FC Barcelona
Nàstic de Tarragona	2-3	RCD Espanyol
Europa	4-2	Badalona
Cornellà	3-4	RCD Espanyol
Sabadell	1-1	Granollers
Girona FC	5-2	Olot
FC Barcelona	5-2	Nàstic de Tarragona
Nàstic de Tarragona	2-5	Figueres
Vilafranca	2-0	FC Barcelona
Europa	2-3	Lleida Esportiu
RCD Espanyol	5-2	Olot
Lleida Esportiu	0-1	Nàstic de Tarragona
Prat	1-0	Olot
Terrassa	5-2	Europa
Llagostera	0-5	Nàstic de Tarragona
Nàstic de Tarragona	3-2	Reus Deportiu
Reus Deportiu	1-4	Llagostera
Sabadell	0-4	Badalona
Vilafranca	2-1	Lleida Esportiu
Llagostera	0-2	Terrassa
Girona FC	5-5	Vilafranca
Granollers	4-4	Olot
Girona FC	3-0	Cerdanyola
FC Barcelona	1-1	Cerdanyola
Badalona	5-5	Reus Deportiu
Cornellà	4-4	Figueres
Girona FC	5-4	Figueres
Sabadell	5-3	FC Barcelona
Nàstic de Tarragona	0-4	Girona FC
Terrassa	3-2	Reus Deportiu
Reus Deportiu	1-1	Europa
Terrassa	5-5	Llagostera
Prat	4-1	Cornellà
Sant Andreu	3-0	Prat
Prat	1-3	Cerdanyola
Sabadell	5-0	Figueres
Olot	1-1	Manlleu
Reus Deportiu	0-1	Terrassa
Sant Andreu	2-2	Terrassa
Nàstic de Tarragona	2-4	Sabadell
Llagostera	3-5	Cerdanyola
Granollers	3-1	Badalona
Figueres	4-0	Reus Deportiu
Europa	4-5	Llagostera
Girona FC	2-4	Terrassa
Olot	1-2	Granollers
Vilafranca	5-5	Olot
RCD Espanyol	4-2	Figueres
Terrassa	3-3	Figueres
Girona FC	5-2	Sabadell
Sabadell	3-3	Cornellà
Granollers	0-2	Girona FC
Olot	0-4	FC Barcelona
Europa	4-4	Girona FC
Lleida Esportiu	5-4	Vilafranca
Sant Andreu	5-2	Olot
Lleida Esportiu	2-2	Granollers
Llagostera	4-2	Granollers
Terrassa	0-2	Llagostera
Lleida Esportiu	5-3	Badalona
Sabadell	3-5	Figueres
Nàstic de Tarragona	3-4	Sabadell
Europa	0-5	Cerdanyola
Cornellà	4-5	Manlleu
Reus Deportiu	5-1	Lleida Esportiu
Vilafranca	1-0	RCD Espanyol
RCD Espanyol	4-5	Girona FC
Granollers	4-0	Vilafranca
Sant Andreu	2-4	Reus Deportiu
Lleida Esportiu	1-2	FC Barcelona
Llagostera	4-4	Sant Andreu
Girona FC	2-4	Llagostera
Figueres	0-4	Reus Deportiu
Lleida Esportiu	1-5	RCD Espanyol
Prat	2-0	Sant Andreu
Badalona	0-1	Sant Andreu
Olot	1-2	Reus Deportiu
Sabadell	1-4	Sant Andreu
Sant Andreu	0-1	Lleida Esportiu
FC Barcelona	3-2	Lleida Esportiu
Badalona	3-0	Llagostera
Llagostera	0-2	Cornellà
Cornellà	3-1	Olot
Nàstic de Tarragona	0-1	Cornellà
Girona FC	4-1	RCD Espanyol
Vilafranca	4-4	Llagostera
Nàstic de Tarragona	3-4	FC Barcelona
Nàstic de Tarragona	0-2	Cornellà
Terrassa	0-0	Manlleu
Sabadell	3-3	Terrassa
Nàstic de Tarragona	2-2	Vilafranca
Sabadell	4-4	Figueres
RCD Espanyol	2-0	Prat
Cornellà	1-2	Terrassa
Prat	0-0	Badalona
Figueres	0-2	Granollers
Manlleu	5-0	FC Barcelona
Sant Andreu	3-5	Reus Deportiu
RCD Espanyol	2-1	Badalona
Sant Andreu	5-0	Cornellà
Olot	4-5	FC Barcelona
Olot	5-1	Figueres
Prat	3-1	Cornellà
Olot	3-1	Figueres
Olot	2-4	Girona FC
Sabadell	5-3	Cerdanyola
Llagostera	3-5	Manlleu
Granollers	2-4	Badalona
RCD Espanyol	4-5	Cerdanyola
Llagostera	4-4	Sant Andreu
FC Barcelona	5-3	Vilafranca
Lleida Esportiu	2-1	Reus Deportiu
Granollers	2-1	Cerdanyola
Europa	0-3	Manlleu
Granollers	4-5	RCD Espanyol
FC Barcelona	4-5	Nàstic de Tarragona
Manlleu	3-3	RCD Espanyol
Sabadell	4-5	Manlleu
Europa	2-0	Reus Deportiu
Olot	4-0	Granollers
Terrassa	3-1	RCD Espanyol
Vilafranca	2-5	Terrassa
Reus Deportiu	0-5	Sant Andreu
Nàstic de Tarragona	4-0	RCD Espanyol
Prat	4-4	Sant Andreu
Terrassa	5-3	Vilafranca
FC Barcelona	3-0	Llagostera
Granollers	5-0	Olot
Figueres	4-5	Vilafranca
Terrassa	3-3	Llagostera
Prat	3-3	RCD Espanyol
Nàstic de Tarragona	1-4	Llagostera
Olot	4-4	Vilafranca
Olot	0-3	Granollers
Badalona	3-3	Llagostera
Europa	1-5	Olot
Cornellà	2-2	Sant Andreu
Girona FC	5-1	Lleida Esportiu
Reus Deportiu	5-2	Sant Andreu
Vilafranca	0-4	Lleida Esportiu
FC Barcelona	1-5	Llagostera
Badalona	3-4	Terrassa
FC Barcelona	1-1	RCD Espanyol
Sant Andreu	5-4	Terrassa
Sabadell	0-1	Girona FC
RCD Espanyol	1-3	Prat
Nàstic de Tarragona	3-3	Prat
FC Barcelona	5-2	Prat
RCD Espanyol	1-4	Cerdanyola
Terrassa	1-2	Sant Andreu
Llagostera	5-4	Europa
Lleida Esportiu	1-5	Figueres
Nàstic de Tarragona	1-3	Girona FC
Cornellà	5-2	Badalona
Olot	3-0	Lleida Esportiu
Granollers	5-3	Manlleu
Granollers	1-3	Llagostera
Sabadell	5-3	Olot
Reus Deportiu	3-1	Terrassa
Prat	3-3	RCD Espanyol
RCD Espanyol	5-1	Nàstic de Tarragona
Sant Andreu	0-0	Terrassa
Cornellà	5-0	Terrassa
Vilafranca	4-5	Cornellà
Vilafranca	4-5	Badalona
FC Barcelona	4-3	Lleida Esportiu
Terrassa	5-4	Cornellà
Olot	1-5	Europa
Lleida Esportiu	4-5	Figueres
Sant Andreu	5-0	Europa
Sant Andreu	3-1	Girona FC
Sabadell	1-3	Llagostera
Nàstic de Tarragona	1-3	Olot
Badalona	5-4	Prat
Olot	4-5	Llagostera
FC Barcelona	3-1	Cornellà
Olot	0-5	Granollers
"""


def main_analitica_deportiva():
    equipos_master = procesar_archivo_liga(historico_partidos_txt)

    goleador, goleado = obtener_extremos_goleadores(equipos_master)

    print("==========================================================================")
    print("           RESPUESTAS DE NEGOCIO - DEPARTAMENTO DE ANALÍTICA")
    print("==========================================================================")
    print(f"» 1 & 3. Equipo MÁS Goleador y total marcado:\n"
          f"         -> {goleador[0]} con un total de {goleador[1]['gols_favor']} goles a favor.\n")
    print(f"» 2. Equipo MÁS Goleado:\n"
          f"         -> {goleado[0]} con un total de {goleado[1]['gols_contra']} goles en contra.\n")

    mostrar_tabla_clasificacion(equipos_master)

main_analitica_deportiva()

           RESPUESTAS DE NEGOCIO - DEPARTAMENTO DE ANALÍTICA
» 1 & 3. Equipo MÁS Goleador y total marcado:
         -> Vilafranca con un total de 107 goles a favor.

» 2. Equipo MÁS Goleado:
         -> Vilafranca con un total de 112 goles en contra.


         CLASIFICACIÓN FINAL DE LA LIGA CATALANA - REPORTE DE ANALÍTICA
equip                | gols_favor | gols_contra | guanyats | empatats | perduts | punts
------------------------------------------------------------------------------------------
Terrassa             | 96         | 97          | 15       | 10       | 12      | 55
Girona FC            | 83         | 60          | 17       | 3        | 7       | 54
Llagostera           | 88         | 91          | 14       | 6        | 13      | 48
Badalona             | 99         | 95          | 13       | 7        | 13      | 46
Granollers           | 72         | 71          | 14       | 3        | 13      | 45
Cornellà             | 80         | 72          | 13       | 5        |